<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 12: Transfer Learning

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 12 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta12/hafta12_transfer_learning.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta12/hafta12_transfer_learning.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 12 - Transfer Öğrenme (Transfer Learning)

Bu defterde önceden eğitilmiş modelleri kullanarak kedi/köpek sınıflandırması yapacağız.

## İçerik
1. Transfer Öğrenme Nedir?
2. Veri Hazırlığı (CIFAR-10 Kedi vs Köpek Alt Kümesi)
3. Transfer Öğrenme Olmadan CNN
4. MobileNetV2 ile Transfer Öğrenme
5. Karşılaştırma ve Sonuçlar

## 1. Transfer Öğrenme Nedir?

**Transfer Öğrenme**, büyük bir veri seti üzerinde eğitilmiş bir modelin öğrendiği bilgileri yeni bir göreve aktarma tekniğidir.

### Neden Transfer Öğrenme?

| Avantaj | Açıklama |
|---------|----------|
| **Az veri** | Küçük veri setleriyle bile yüksek doğruluk elde edilir |
| **Hızlı eğitim** | Sıfırdan eğitmekten çok daha hızlıdır |
| **Daha iyi özellikler** | ImageNet üzerinde öğrenilen özellikler çok güçlüdür |
| **Kaynak tasarrufu** | Daha az GPU/CPU kaynağı gerektirir |

### Nasıl Çalışır?

```
┌─────────────────────────────┐
│   Önceden Eğitilmiş Model   │  ← ImageNet'te 1000 sınıf üzerinde eğitilmiş
│   (MobileNetV2 - donmuş)    │  ← Bu katmanlar güncellenmez (freeze)
├─────────────────────────────┤
│   GlobalAveragePooling2D    │  ← Yeni eklenen katmanlar
│   Dense(128, relu)          │  ← Sadece bu katmanlar eğitilir
│   Dense(1, sigmoid)         │  ← Kedi/Köpek çıktısı
└─────────────────────────────┘
```

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow sürümü: {tf.__version__}")

## 2. Veri Hazırlığı

CIFAR-10 veri setinden **Kedi (sınıf 3)** ve **Köpek (sınıf 5)** alt kümesini çıkaracağız.

In [ ]:
# CIFAR-10 veri setini yükle
(X_train_full, y_train_full), (X_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()

# Kedi (3) ve Köpek (5) sınıflarını filtrele
kedi_sinifi = 3
kopek_sinifi = 5

# Eğitim seti
train_mask = np.isin(y_train_full.flatten(), [kedi_sinifi, kopek_sinifi])
X_train = X_train_full[train_mask]
y_train = (y_train_full[train_mask].flatten() == kopek_sinifi).astype(np.float32)  # Kedi=0, Köpek=1

# Test seti
test_mask = np.isin(y_test_full.flatten(), [kedi_sinifi, kopek_sinifi])
X_test = X_test_full[test_mask]
y_test = (y_test_full[test_mask].flatten() == kopek_sinifi).astype(np.float32)

# Normalizasyon
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

print(f"Eğitim seti: {X_train.shape[0]} görüntü")
print(f"Test seti: {X_test.shape[0]} görüntü")
print(f"Kedi sayısı (eğitim): {(y_train == 0).sum()}")
print(f"Köpek sayısı (eğitim): {(y_train == 1).sum()}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Örnek görüntüleri göster
etiketler = ['Kedi', 'Köpek']

plt.figure(figsize=(15, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[i])
    plt.title(etiketler[int(y_train[i])], fontsize=10)
    plt.axis('off')

plt.suptitle('Kedi vs Köpek Örnekleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Transfer Öğrenme Olmadan CNN (Karşılaştırma İçin)

In [ ]:
# Basit CNN modeli (sıfırdan eğitim)
basit_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

basit_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Basit CNN Modeli:")
print(f"Toplam parametre: {basit_model.count_params():,}")

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Basit modeli eğit
gecmis_basit = basit_model.fit(
    X_train_norm, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_norm, y_test),
    verbose=1
)

basit_kayip, basit_dogruluk = basit_model.evaluate(X_test_norm, y_test, verbose=0)
print(f"\nBasit CNN Test Doğruluğu: {basit_dogruluk*100:.2f}%")

## 4. MobileNetV2 ile Transfer Öğrenme

### Görüntüleri 128x128'e yeniden boyutlandır (MobileNetV2 için)

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Görüntüleri 128x128'e yeniden boyutlandır (MobileNetV2 için)
X_train_resized = tf.image.resize(X_train_norm, (128, 128)).numpy()
X_test_resized = tf.image.resize(X_test_norm, (128, 128)).numpy()

print(f"Yeniden boyutlandırılmış eğitim seti: {X_train_resized.shape}")
print(f"Yeniden boyutlandırılmış test seti: {X_test_resized.shape}")

### MobileNetV2 önceden eğitilmiş modelini yükle

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# MobileNetV2 önceden eğitilmiş modelini yükle
taban_model = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

# Taban modelin katmanlarını dondur (eğitilmesini engelle)
taban_model.trainable = False

print(f"MobileNetV2 katman sayısı: {len(taban_model.layers)}")
print(f"Eğitilebilir parametre: {sum(tf.keras.backend.count_params(w) for w in taban_model.trainable_weights):,}")
print(f"Donmuş parametre: {sum(tf.keras.backend.count_params(w) for w in taban_model.non_trainable_weights):,}")

### Sinir Ağı Modeli Oluşturma

Keras Sequential API ile katman katman sinir ağı modeli inşa ediyoruz. Her katmanın kendine özgü bir görevi vardır (özellik çıkarma, boyut düşürme, sınıflandırma).

In [ ]:
# Transfer öğrenme modeli oluştur
transfer_model = tf.keras.Sequential([
    taban_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

transfer_model.summary()

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Transfer öğrenme modelini eğit
gecmis_transfer = transfer_model.fit(
    X_train_resized, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_resized, y_test),
    verbose=1
)

transfer_kayip, transfer_dogruluk = transfer_model.evaluate(X_test_resized, y_test, verbose=0)
print(f"\nTransfer Öğrenme Test Doğruluğu: {transfer_dogruluk*100:.2f}%")

## 5. Karşılaştırma ve Sonuçlar

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Doğruluk karşılaştırması
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eğitim doğruluğu
axes[0].plot(gecmis_basit.history['val_accuracy'], label='Basit CNN', linewidth=2)
axes[0].plot(gecmis_transfer.history['val_accuracy'], label='Transfer Öğrenme', linewidth=2)
axes[0].set_title('Doğrulama Doğruluğu Karşılaştırması', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Doğruluk')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Kayıp karşılaştırması
axes[1].plot(gecmis_basit.history['val_loss'], label='Basit CNN', linewidth=2)
axes[1].plot(gecmis_transfer.history['val_loss'], label='Transfer Öğrenme', linewidth=2)
axes[1].set_title('Doğrulama Kaybı Karşılaştırması', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Kayıp')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Sonuç tablosu

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Sonuç tablosu
print("=" * 50)
print("MODEL KARŞILAŞTIRMASI")
print("=" * 50)
print(f"{'Model':<25} {'Test Doğruluğu':>15}")
print("-" * 50)
print(f"{'Basit CNN':<25} {basit_dogruluk*100:>14.2f}%")
print(f"{'Transfer Öğrenme':<25} {transfer_dogruluk*100:>14.2f}%")
print("=" * 50)

fark = (transfer_dogruluk - basit_dogruluk) * 100
print(f"\nTransfer öğrenme, basit CNN'e göre %{fark:.2f} {'daha iyi' if fark > 0 else 'daha kötü'} performans gösterdi.")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Tahminleri görselleştir
tahminler = transfer_model.predict(X_test_resized[:20])

plt.figure(figsize=(15, 6))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_test[i])
    
    tahmin = 'Köpek' if tahminler[i] > 0.5 else 'Kedi'
    gercek = etiketler[int(y_test[i])]
    guvence = tahminler[i][0] if tahminler[i] > 0.5 else 1 - tahminler[i][0]
    
    renk = 'green' if tahmin == gercek else 'red'
    plt.title(f"{tahmin}\n%{guvence*100:.0f}", fontsize=8, color=renk)
    plt.axis('off')

plt.suptitle('Transfer Öğrenme Tahminleri (Yeşil=Doğru, Kırmızı=Yanlış)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Özet

Bu defterde öğrendiklerimiz:

1. **Transfer Öğrenme**: Önceden eğitilmiş modellerin bilgisini yeni görevlere aktarma
2. **MobileNetV2**: Hafif ve verimli önceden eğitilmiş model
3. **Katman Dondurma**: Taban modelin ağırlıklarını sabit tutma
4. **Özel Üst Katmanlar**: Yeni görev için sınıflandırma katmanları ekleme
5. **Performans Karşılaştırması**: Transfer öğrenme genellikle sıfırdan eğitimden daha iyi sonuç verir

### Önemli Noktalar
- Az veri olduğunda transfer öğrenme çok etkilidir
- Taban modeli dondurarak sadece üst katmanlar eğitilir
- Fine-tuning ile taban modelin son katmanları da açılarak ince ayar yapılabilir

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>